## 1. Reading a table directly from a page using pandas

`pd.read_html()` sends a request to the URL, fetches the raw HTML, and looks for any `<table>` tags on the page.
It returns a list of DataFrames, one per table found, so `tables[0]` gives you the first one, ready to work with.

In [ ]:
# If you didn't install these libraries, run this cell (remove the # below):
#!pip install pandas requests

In [ ]:
import pandas as pd

In [ ]:
url = "https://www.paragraf.rs/statistika/minimalna_zarada.html"

In [ ]:
tables = pd.read_html(url)
wages = tables[0]

wages.head()

In [ ]:
wages.to_excel("minimal wages serbia.xlsx", index=False)

## 2. Downloading raw HTML and parsing it into a dataframe

Some websites block automated requests or rely on JavaScript to render their content, making it impossible to scrape them directly from Python. 

In these cases, you can save the page manually using your browser's "Save As" function, which captures the raw HTML exactly as it appears in your browser. 

You can then load that file locally in Python and use BeautifulSoup to parse it and extract whatever you need> titles, links, dates, or any other structured information buried in the HTML. This is a slower, more manual approach, but it works on pages that would otherwise be impossible to scrape, and it is especially useful for one-off data collection tasks where you only need to do it once.

In [ ]:
import json
import re
import pandas as pd
from datetime import datetime
from pathlib import Path

In [ ]:
records = []

for filepath in Path("htmls").glob("*.html"):
    with open(filepath, "r", encoding="utf-8") as f:
        html = f.read()

    match = re.search(r'window\.SERVER_DATA\s*=\s*(\{.*?\});\s*</script>', html, re.DOTALL)
    if not match:
        print(f"Skipped (no JSON): {filepath.name}")
        continue

    data = json.loads(match.group(1))
    items = data["NodeLoader"]["node"]["lists"][0]["items"]

    for item in items:
        records.append({
            "title": item["title"],
            "date": datetime.fromtimestamp(item["published"]).strftime("%Y-%m-%d"),
            "url": "https://www.europol.europa.eu" + item["alias"],
            "source_file": filepath.name
        })

europol = pd.DataFrame(records)
europol.head()

In [ ]:
europol.shape

In [ ]:
europol.to_excel("europol news.xlsx", index=False)

## 3. Pulling data through network requests

When you browse a website, your browser is constantly making requests in the background to fetch data, and you can spy on these using the Network tab in DevTools. 

Some websites, like the FIFA rankings page, load their data through undocumented internal APIs that return clean JSON, even though there is no official documentation or download button anywhere on the site. 

To replicate one of these requests in Python, open DevTools, find the relevant request, copy it as cURL, and paste it into [curlconverter.com](https://curlconverter.com/python/) to get a working Python snippet. Note that this approach does not work on every website, some use authentication tokens that expire quickly, others actively block scrapers, but when it does work, it is often the fastest way to get clean, structured data.

In [ ]:
import requests
import pandas as pd

In [ ]:
headers = {
    'Accept': '*/*',
    'Accept-Language': 'en-GB,en-US;q=0.9,en;q=0.8,sr;q=0.7',
    'Connection': 'keep-alive',
    'Origin': 'https://inside.fifa.com',
    'Referer': 'https://inside.fifa.com/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site',
    'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Mobile Safari/537.36',
    'sec-ch-ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    'sec-ch-ua-mobile': '?1',
    'sec-ch-ua-platform': '"Android"',
}

params = {
    'rankingScheduleId': 'FRS_Male_Football_20260119',
    'language': 'en',
}

response = requests.get('https://api.fifa.com/api/v3/fifarankings/rankings/rankingsbyschedule', params=params, headers=headers)

In [ ]:
response.json()

In [ ]:
data = response.json()

men = pd.DataFrame([{
    "rank": team["Rank"],
    "country": team["TeamName"][0]["Description"],
    "confederation": team["ConfederationName"],
    "points": round(team["TotalPoints"], 2),
    "previous_points": round(team["PrevPoints"], 2),
    "prev_rank": team["PrevRank"],
    "rated_matches": team["RatedMatches"],
    "changes": team["RankingMovement"]
} for team in data["Results"]])

men.head(10)

In [ ]:
men.to_excel("fifa rankings.xlsx", index=False)